# Collect Cosmos-Policy contrastive input pairs across 3 object-pair scenes

Sister notebook to `collect_policy_inputs_milk.ipynb`. Builds **paired**
policy-model inputs for downstream contrastive-direction analysis (LQR /
SVD), but sweeps the **3 successful uncluttered scenes** from
[notebooks/stress_test/10_object_pair_selected](../../stress_test/10_object_pair_selected.ipynb)
instead of a single milk prompt:

| cfg | goal pair                     | prompt                                                       |
|-----|-------------------------------|--------------------------------------------------------------|
| C01 | tomato_sauce + milk           | put both the tomato sauce and the milk in the basket         |
| C03 | cream_cheese + tomato_sauce   | put both the cream cheese and the tomato sauce in the basket |
| C05 | milk + butter                 | put both the milk and the butter in the basket               |

For each config we only roll out **the episode indices that succeeded** in
the uncluttered scene (parsed from `ep##--SUCCESS.mp4` filenames under
`notebooks/stress_test/rollouts/10_object_pair_selected/<cfg>/`).

Per config we run two paired-rollout campaigns:

1. **neg-drives** — env_neg (cluttered, full 8-object scene) steps; env_pos
   (uncluttered) is state-injected via `SceneRetargetTask.set_init_state`.
2. **pos-drives** — env_pos steps; env_neg is state-injected via
   `_expand_pos_to_neg` (kept-body qpos read at env_pos's joint addresses,
   removed-body qpos pulled from the saved libero init_state).

Differences from the milk notebook:

* **SceneRetargetTask, not SceneRemoveObjects** — matches the BDDL used to
  generate the rollouts under `10_object_pair_selected/`. Different `:goal`
  rewrite, different `:obj_of_interest`. With `replacements=()` it
  degenerates to REMOVE + RETARGET on the original 8-object BDDL.
* **Cluttered negative also rewrites goal + prompt** — so neg-drives
  terminates on real success instead of running to `max_env_steps`.
* **`_expand_pos_to_neg` reads/writes via `sim.model.get_joint_qpos_addr`**
  — the milk version assumes sequential layout (`1+robot_nq + 7*kept_idx`),
  which only holds for SceneRemoveObjects' sequential writer.
* **Per-row `config_idx`** plus a `prompts` lookup table in each NPZ.

Output:

```
notebooks/lqr/inputs/policy_inputs/libero_10__task00__object_pairs_pos_neg/
  positive.npz   # uncluttered renders at every captured pose
  negative.npz   # cluttered  renders at the same poses
  manifest.json  # configs, prompts, per-rollout summaries
  derived__<slug>.bddl   # BDDLs emitted by SceneRetargetTask, one per (cfg, role)
```


In [ ]:
# notebooks/_setup.py lives two dirs up (notebooks/lqr/inputs/<this>.ipynb).
import sys; sys.path.insert(0, '../..')
from _setup import setup_env
setup_env()

import os
os.environ.setdefault('MUJOCO_GL', 'egl')
os.environ.setdefault('PYOPENGL_PLATFORM', 'egl')


In [ ]:
import copy
import hashlib
import json
import re
import time
from collections import deque
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional, Tuple

import numpy as np

from libero.libero import benchmark, get_libero_path
from cosmos_policy.experiments.robot.libero.libero_utils import (
    get_libero_env, get_libero_dummy_action,
)
from cosmos_policy.experiments.robot.libero.run_libero_eval import (
    PolicyEvalConfig, prepare_observation, TASK_MAX_STEPS,
)
from cosmos_policy.experiments.robot.cosmos_utils import (
    get_action, get_model, load_dataset_stats, init_t5_text_embeddings_cache,
)


## 1. `SceneRetargetTask` (copied from `stress_test/10_object_pair_selected.ipynb`)

A unified BDDL rewriter that composes RENAME + REMOVE + RETARGET passes. Here
we use it with `replacements=()`, so it degenerates to REMOVE (slots not in
`keep_only`) + RETARGET (`:goal` rewritten to require both `goal_pair` keys in
`<container_key>_contain_region`, `:obj_of_interest` regenerated,
`(:language ...)` swapped to `prompt`).

We use it for **both** the uncluttered (positive) and cluttered (negative)
envs per config so the only difference between them is `keep_only` — same
goal predicate, same prompt, same `:obj_of_interest`. That lets the
cluttered neg-drives campaign terminate on real success instead of running
to `max_env_steps`.


In [ ]:
def _find_section_bounds(bddl, keyword):
    needle = f'(:{keyword}'
    start = bddl.find(needle)
    if start == -1:
        return None
    depth = 0
    for i in range(start, len(bddl)):
        c = bddl[i]
        if c == '(':
            depth += 1
        elif c == ')':
            depth -= 1
            if depth == 0:
                return start, i + 1
    return None


def _drop_section_lines(bddl, keyword, predicate):
    bounds = _find_section_bounds(bddl, keyword)
    if bounds is None:
        return bddl
    s, e = bounds
    section = bddl[s:e]
    keep = [line for line in section.split('\n') if not predicate(line.strip())]
    return bddl[:s] + '\n'.join(keep) + bddl[e:]


def _parse_objects_order(bddl):
    bounds = _find_section_bounds(bddl, 'objects')
    if bounds is None:
        raise ValueError('BDDL missing :objects section')
    s, e = bounds
    section = bddl[s:e]
    keys = []
    for line in section.split('\n'):
        stripped = line.strip()
        if not stripped or stripped.startswith('(:') or stripped == ')':
            continue
        m = re.match(r'(\S+)\s*-\s*\S+', stripped)
        if m:
            keys.append(m.group(1))
    return keys


def _base_name(key):
    return re.sub(r'_\d+$', '', key)


def _strip_paren_block(text, opening_token):
    idx = text.find(opening_token)
    if idx == -1:
        return text
    depth = 0
    end = None
    for i in range(idx, len(text)):
        c = text[i]
        if c == '(':
            depth += 1
        elif c == ')':
            depth -= 1
            if depth == 0:
                end = i + 1
                break
    if end is None:
        return text
    while end < len(text) and text[end] in ' \t':
        end += 1
    if end < len(text) and text[end] == '\n':
        end += 1
    line_start = text.rfind('\n', 0, idx) + 1
    if text[line_start:idx].strip() == '':
        idx = line_start
    return text[:idx] + text[end:]


def _resolve_libero_problem_obj(env):
    cur, seen = env, set()
    for _ in range(8):
        if cur is None or id(cur) in seen:
            break
        seen.add(id(cur))
        if hasattr(cur, 'objects_dict') and hasattr(cur, 'fixtures_dict'):
            return cur
        cur = getattr(cur, 'env', None)
    return None


class StressTest:
    slug: str = 'stock'
    def transform_task(self, task, output_dir=None): return task
    def set_init_state(self, env, init_state, episode_idx=0): return env.set_init_state(init_state)
    def apply_to_env(self, env, episode_idx=0) -> None: return None
    def transform_task_desc(self, desc, env=None): return desc
    def manifest(self) -> dict: return {'kind': type(self).__name__, 'slug': self.slug}


@dataclass
class SceneRetargetTask(StressTest):
    replacements: Tuple[Tuple[str, str, str], ...] = ()
    keep_only: Optional[Tuple[str, ...]] = None
    goal_pair: Tuple[str, str] = ('', '')
    container_key: str = 'basket_1'
    prompt: str = ''
    name_hint: str = 'retarget'

    _bddl_path: Optional[str] = field(default=None, init=False, repr=False)
    _orig_order: Tuple[str, ...] = field(default=(), init=False, repr=False)
    _final_order: Tuple[str, ...] = field(default=(), init=False, repr=False)
    _final_source: Dict[str, str] = field(default_factory=dict, init=False, repr=False)
    _renamed_set: Tuple[str, ...] = field(default=(), init=False, repr=False)

    @property
    def slug(self) -> str:
        payload = (
            'r:' + '|'.join(f'{a}=>{b}:{t}' for a, b, t in self.replacements) +
            '||k:' + (','.join(sorted(self.keep_only)) if self.keep_only is not None else 'ALL') +
            '||g:' + ','.join(self.goal_pair) +
            '||c:' + self.container_key +
            '||p:' + self.prompt
        )
        h = hashlib.md5(payload.encode()).hexdigest()[:6]
        return f'{self.name_hint}_{h}'

    def transform_task(self, task, output_dir=None):
        src_path = os.path.join(
            get_libero_path('bddl_files'), task.problem_folder, task.bddl_file,
        )
        with open(src_path) as f:
            bddl = f.read()

        self._orig_order = tuple(_parse_objects_order(bddl))
        rename_map = {old: (new, ntype) for old, new, ntype in self.replacements}
        self._renamed_set = tuple(new for _, new, _ in self.replacements)

        missing = set(rename_map) - set(self._orig_order)
        if missing:
            raise ValueError(f'replacement old_keys not in :objects: {sorted(missing)}')

        bounds = _find_section_bounds(bddl, 'objects')
        if bounds is not None:
            s, e = bounds
            section = bddl[s:e]
            lines_out = []
            for line in section.split('\n'):
                stripped = line.strip()
                m = re.match(r'(\S+)\s*-\s*\S+', stripped)
                if m and m.group(1) in rename_map:
                    new, ntype = rename_map[m.group(1)]
                    indent = line[:len(line) - len(line.lstrip())]
                    lines_out.append(f'{indent}{new} - {ntype}')
                else:
                    lines_out.append(line)
            bddl = bddl[:s] + '\n'.join(lines_out) + bddl[e:]

        bounds = _find_section_bounds(bddl, 'init')
        if bounds is not None:
            s, e = bounds
            section = bddl[s:e]
            for old, new, _ in self.replacements:
                old_base = _base_name(old)
                section = re.sub(
                    rf'\(On\s+{re.escape(old)}\s+(\w+?)_{re.escape(old_base)}_init_region\)',
                    lambda m, nk=new: f'(On {nk} {m.group(1)}_{nk}_init_region)',
                    section,
                )
            bddl = bddl[:s] + section + bddl[e:]

        bounds = _find_section_bounds(bddl, 'regions')
        if bounds is not None:
            s, e = bounds
            section = bddl[s:e]
            for old, new, _ in self.replacements:
                old_base = _base_name(old)
                section = section.replace(
                    f'({old_base}_init_region',
                    f'({new}_init_region',
                    1,
                )
            bddl = bddl[:s] + section + bddl[e:]

        post_rename_for_orig = {
            ok: (rename_map[ok][0] if ok in rename_map else ok)
            for ok in self._orig_order
        }
        post_rename_keys = [post_rename_for_orig[ok] for ok in self._orig_order]

        if self.keep_only is None:
            keep_set = set(post_rename_keys)
        else:
            keep_set = set(self.keep_only)
            missing_keep = {*self.goal_pair, self.container_key} - keep_set
            if missing_keep:
                raise ValueError(
                    f'keep_only is missing required keys: {sorted(missing_keep)}'
                )

        remove_origs = [ok for ok in self._orig_order if post_rename_for_orig[ok] not in keep_set]
        remove_posts = [post_rename_for_orig[ok] for ok in remove_origs]

        if remove_posts:
            bddl = _drop_section_lines(
                bddl, 'objects',
                lambda L: any(re.match(rf'{re.escape(k)}\s*-', L) for k in remove_posts),
            )
            bddl = _drop_section_lines(
                bddl, 'obj_of_interest',
                lambda L: L in remove_posts,
            )
            bddl = _drop_section_lines(
                bddl, 'init',
                lambda L: any(L.startswith(f'(On {k} ') for k in remove_posts),
            )
            bounds = _find_section_bounds(bddl, 'regions')
            if bounds is not None:
                s, e = bounds
                section = bddl[s:e]
                for ok, pk in zip(remove_origs, remove_posts):
                    region_id = pk if ok in rename_map else _base_name(ok)
                    section = _strip_paren_block(section, f'({region_id}_init_region')
                bddl = bddl[:s] + section + bddl[e:]

        bounds = _find_section_bounds(bddl, 'goal')
        if bounds is not None:
            s, e = bounds
            a, b = self.goal_pair
            new_goal = (
                '(:goal\n'
                f'    (And (In {a} {self.container_key}_contain_region) '
                f'(In {b} {self.container_key}_contain_region))\n'
                '  )'
            )
            bddl = bddl[:s] + new_goal + bddl[e:]

        bounds = _find_section_bounds(bddl, 'obj_of_interest')
        if bounds is not None:
            s, e = bounds
            new_section = (
                '(:obj_of_interest\n'
                f'    {self.goal_pair[0]}\n'
                f'    {self.goal_pair[1]}\n'
                f'    {self.container_key}\n'
                '  )'
            )
            bddl = bddl[:s] + new_section + bddl[e:]

        bddl = re.sub(
            r'\(:language[^)\n]*\)',
            f'(:language {self.prompt})',
            bddl, count=1,
        )

        self._final_order = tuple(_parse_objects_order(bddl))
        self._final_source = {
            pk: ok for ok, pk in zip(self._orig_order, post_rename_keys)
            if pk in keep_set
        }

        out_dir = Path(output_dir) if output_dir else Path('/tmp')
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f'derived__{self.slug}.bddl'
        out_path.write_text(bddl)
        self._bddl_path = str(out_path.resolve())

        if hasattr(task, '_replace'):
            return task._replace(bddl_file=self._bddl_path, problem_folder='')
        new_task = copy.copy(task)
        new_task.bddl_file = self._bddl_path
        new_task.problem_folder = ''
        return new_task

    def set_init_state(self, env, init_state, episode_idx=0):
        problem = _resolve_libero_problem_obj(env)
        if problem is None:
            return env.set_init_state(init_state)
        sim = problem.sim
        nq, nv = int(sim.model.nq), int(sim.model.nv)
        n_orig = len(self._orig_order)
        n_final = len(self._final_order)
        robot_nq = nq - 7 * n_final
        robot_nv = nv - 6 * n_final
        src_nq = robot_nq + 7 * n_orig
        src_nv = robot_nv + 6 * n_orig
        expected_src_len = 1 + src_nq + src_nv
        if len(init_state) != expected_src_len:
            raise ValueError(
                f'SceneRetargetTask.set_init_state: saved init_state size '
                f'{len(init_state)} != expected {expected_src_len}'
            )

        cur = sim.get_state()
        new_qpos = cur.qpos.copy()
        new_qvel = cur.qvel.copy()

        new_qpos[:robot_nq] = init_state[1 : 1 + robot_nq]
        new_qvel[:robot_nv] = init_state[1 + src_nq : 1 + src_nq + robot_nv]

        renamed_set = set(self._renamed_set)
        for final_key in self._final_order:
            source_key = self._final_source[final_key]
            j = self._orig_order.index(source_key)
            src_qpos_start = 1 + robot_nq + 7 * j
            src_qvel_start = 1 + src_nq + robot_nv + 6 * j

            mjobj = problem.objects_dict.get(final_key)
            if mjobj is None or not getattr(mjobj, 'joints', None):
                continue
            addr = sim.model.get_joint_qpos_addr(mjobj.joints[0])
            qstart = addr[0] if isinstance(addr, tuple) else int(addr)
            vaddr = sim.model.get_joint_qvel_addr(mjobj.joints[0])
            vstart = vaddr[0] if isinstance(vaddr, tuple) else int(vaddr)

            new_qpos[qstart : qstart + 7] = init_state[src_qpos_start : src_qpos_start + 7]
            new_qvel[vstart : vstart + 6] = init_state[src_qvel_start : src_qvel_start + 6]

            if final_key in renamed_set:
                new_qpos[qstart + 2] = cur.qpos[qstart + 2]
                new_qpos[qstart + 3 : qstart + 7] = cur.qpos[qstart + 3 : qstart + 7]
                new_qvel[vstart : vstart + 6] = 0.0

        flat = np.concatenate([[init_state[0]], new_qpos, new_qvel])
        return env.set_init_state(flat)

    def transform_task_desc(self, desc, env=None):
        return self.prompt or desc

    def manifest(self) -> dict:
        return {
            'kind': 'SceneRetargetTask',
            'slug': self.slug,
            'name_hint': self.name_hint,
            'replacements': [list(t) for t in self.replacements],
            'keep_only': list(self.keep_only) if self.keep_only is not None else None,
            'goal_pair': list(self.goal_pair),
            'container_key': self.container_key,
            'prompt': self.prompt,
            'orig_order': list(self._orig_order),
            'final_order': list(self._final_order),
            'derived_bddl_path': self._bddl_path,
        }


## 1b. `_expand_pos_to_neg`: inverse of `SceneRetargetTask.set_init_state`

For positive-drives we need the opposite mapping: take **env_pos**'s live
MuJoCo state and produce a flat init_state vector sized for **env_neg** (the
cluttered scene, robot + all 8 original objects).

Unlike the milk version, which assumed a sequential `1 + robot_nq + 7*kept_idx`
layout, this version uses `sim.model.get_joint_qpos_addr` on **both** envs to
look up where each kept body sits. That way the kept-body mapping is
independent of BDDL declaration order or MuJoCo's body-loading sequence.

* Robot qpos/qvel and time come from `env_pos.sim.data`.
* Kept-body qpos/qvel: read at env_pos's joint addresses, written at env_neg's
  joint addresses (matched by body name).
* Removed-body qpos/qvel slots stay at their `init_state_flat` values
  (i.e. the libero episode's saved initial pose) — they never physically
  moved because they don't exist in env_pos.


In [ ]:
def _expand_pos_to_neg(env_pos, env_neg, init_state_flat, pos_stress):
    """Read env_pos's current MuJoCo state and produce a flat init_state for env_neg.
    See markdown above for layout rationale."""
    pos_problem = _resolve_libero_problem_obj(env_pos)
    neg_problem = _resolve_libero_problem_obj(env_neg)
    if pos_problem is None or neg_problem is None:
        raise RuntimeError('could not resolve env_pos / env_neg problem objects')
    pos_sim, neg_sim = pos_problem.sim, neg_problem.sim
    pos_nq, pos_nv = int(pos_sim.model.nq), int(pos_sim.model.nv)
    neg_nq, neg_nv = int(neg_sim.model.nq), int(neg_sim.model.nv)
    n_final = len(pos_stress._final_order)
    n_orig  = len(pos_stress._orig_order)
    robot_nq = pos_nq - 7 * n_final
    robot_nv = pos_nv - 6 * n_final
    if neg_nq != robot_nq + 7 * n_orig:
        raise ValueError(f'neg_nq {neg_nq} != robot_nq+7*n_orig {robot_nq + 7*n_orig}')
    expected_neg_len = 1 + neg_nq + neg_nv
    if len(init_state_flat) != expected_neg_len:
        raise ValueError(
            f'init_state_flat size {len(init_state_flat)} != expected {expected_neg_len}'
        )

    out = np.asarray(init_state_flat, dtype=np.float64).copy()
    pos_data = pos_sim.data
    out[0] = float(pos_data.time)

    pos_qpos = np.asarray(pos_data.qpos)
    pos_qvel = np.asarray(pos_data.qvel)
    out[1 : 1 + robot_nq] = pos_qpos[:robot_nq]
    out[1 + neg_nq : 1 + neg_nq + robot_nv] = pos_qvel[:robot_nv]

    for k in pos_stress._final_order:
        pobj = pos_problem.objects_dict.get(k)
        nobj = neg_problem.objects_dict.get(k)
        if pobj is None or nobj is None or not pobj.joints or not nobj.joints:
            continue
        ap = pos_sim.model.get_joint_qpos_addr(pobj.joints[0])
        qp = ap[0] if isinstance(ap, tuple) else int(ap)
        an = neg_sim.model.get_joint_qpos_addr(nobj.joints[0])
        qn = an[0] if isinstance(an, tuple) else int(an)
        out[1 + qn : 1 + qn + 7] = pos_qpos[qp : qp + 7]

        vap = pos_sim.model.get_joint_qvel_addr(pobj.joints[0])
        vp = vap[0] if isinstance(vap, tuple) else int(vap)
        van = neg_sim.model.get_joint_qvel_addr(nobj.joints[0])
        vn = van[0] if isinstance(van, tuple) else int(van)
        out[1 + neg_nq + vn : 1 + neg_nq + vn + 6] = pos_qvel[vp : vp + 6]

    return out


## 2. Config: 3 scenes × successful libero init-state indices

Each config selects which libero init-state indices to use by scanning the
`ep##--SUCCESS.mp4` filenames under
`notebooks/stress_test/rollouts/10_object_pair_selected/<name>/`.


In [ ]:
SUITE_NAME  = 'libero_10'
TASK_ID     = 0
RESOLUTION  = 256
CONTAINER   = 'basket_1'

ROLLOUTS_ROOT = Path('notebooks/stress_test/rollouts/10_object_pair_selected')

OUT_DIR = Path('notebooks/lqr/inputs/policy_inputs') / f'{SUITE_NAME}__task{TASK_ID:02d}__object_pairs_pos_neg'
OUT_DIR.mkdir(parents=True, exist_ok=True)
POSITIVE_NPZ  = OUT_DIR / 'positive.npz'
NEGATIVE_NPZ  = OUT_DIR / 'negative.npz'
MANIFEST_JSON = OUT_DIR / 'manifest.json'


def _success_episodes_from_dir(scene_dir: Path):
    eps = []
    for p in sorted(scene_dir.glob('ep*--SUCCESS.mp4')):
        m = re.match(r'ep(\d+)--SUCCESS\.mp4', p.name)
        if m:
            eps.append(int(m.group(1)))
    return eps


SCENE_SPECS = [
    {
        'name_hint':  'C01__tomato_sauce__milk__uncluttered',
        'goal_pair':  ('tomato_sauce_1', 'milk_1'),
        'prompt':     'put both the tomato sauce and the milk in the basket',
    },
    {
        'name_hint':  'C03__cream_cheese__tomato_sauce__uncluttered',
        'goal_pair':  ('cream_cheese_1', 'tomato_sauce_1'),
        'prompt':     'put both the cream cheese and the tomato sauce in the basket',
    },
    {
        'name_hint':  'C05__milk__butter__uncluttered',
        'goal_pair':  ('milk_1', 'butter_1'),
        'prompt':     'put both the milk and the butter in the basket',
    },
]

SCENE_CONFIGS = []
for cfg_idx, spec in enumerate(SCENE_SPECS):
    scene_dir = ROLLOUTS_ROOT / spec['name_hint']
    if not scene_dir.exists():
        raise FileNotFoundError(f'missing rollouts dir: {scene_dir.resolve()}')
    succ = _success_episodes_from_dir(scene_dir)
    if not succ:
        raise RuntimeError(f'no SUCCESS episodes parsed from {scene_dir}')
    SCENE_CONFIGS.append({
        **spec,
        'cfg_idx': cfg_idx,
        'episodes': succ,
    })

for c in SCENE_CONFIGS:
    print(f"  cfg{c['cfg_idx']}  {c['name_hint']:50s}  episodes={c['episodes']}")
total_paired_rollouts = sum(len(c['episodes']) for c in SCENE_CONFIGS) * 2
print(f'total paired rollouts: {total_paired_rollouts}  (2 drives x N successful episodes summed across configs)')
print(f'positive -> {POSITIVE_NPZ.resolve()}')
print(f'negative -> {NEGATIVE_NPZ.resolve()}')


## 3. Task suite + init states + model


In [ ]:
task_suite = benchmark.get_benchmark_dict()[SUITE_NAME]()
task = task_suite.get_task(TASK_ID)
init_states = task_suite.get_task_init_states(TASK_ID)
max_env_steps = TASK_MAX_STEPS[SUITE_NAME]
print(f'task.language        : {task.language!r}')
print(f'init states available: {init_states.shape[0]}')
print(f'max_env_steps        : {max_env_steps}')
for c in SCENE_CONFIGS:
    assert all(0 <= ep < init_states.shape[0] for ep in c['episodes']), c


In [ ]:
cfg = PolicyEvalConfig(
    config='cosmos_predict2_2b_480p_libero__inference_only',
    ckpt_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B',
    config_file='cosmos_policy/config/config.py',
    dataset_stats_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_dataset_statistics.json',
    t5_text_embeddings_path='nvidia/Cosmos-Policy-LIBERO-Predict2-2B/libero_t5_embeddings.pkl',
    use_wrist_image=True, use_proprio=True, normalize_proprio=True, unnormalize_actions=True,
    chunk_size=16, num_open_loop_steps=16, trained_with_image_aug=True,
    use_jpeg_compression=True, flip_images=True,
    num_denoising_steps_action=5,
    num_denoising_steps_future_state=1, num_denoising_steps_value=1,
    task_suite_name=SUITE_NAME,
)
dataset_stats = load_dataset_stats(cfg.dataset_stats_path)
init_t5_text_embeddings_cache(cfg.t5_text_embeddings_path)
model, _ = get_model(cfg)
print('model ready')


## 4. Rollout helpers — pos-drives via 2-pass replay, neg-drives via direct paired drive

To reproduce the exact pos-drives trajectories from
[`10_object_pair_selected/`](../../stress_test/10_object_pair_selected.ipynb),
we structure pos-drives as **two passes**:

1. `record_pos_rollout(env_pos, ...)` runs env_pos alone (no env_neg around)
   and records, per inference call, `(sim_state_flat, obs_pos_packed)`. Run
   with the same env-construction + episode-order sequence as the stress
   test, so the policy's torch RNG state evolves identically.
2. `render_pos_records_in_neg(env_pos, env_neg, init_state, ...)` restores
   env_pos to each recorded sim state and renders env_neg via
   `_expand_pos_to_neg`. No policy_fn calls — pure state replay and rendering.

For neg-drives there is no stress-test ground truth to reproduce, so we keep
the original paired-drive logic (`rollout_collect_paired(driver_role='neg', ...)`).


In [ ]:
def policy_fn(obs, desc):
    out = get_action(
        cfg, model, dataset_stats, obs, desc,
        num_denoising_steps_action=cfg.num_denoising_steps_action,
        generate_future_state_and_value_in_parallel=True,
    )
    return out['actions']


def _store_obs(obs_packed):
    """Strip prepare_observation's dict down to the fields we save in the NPZ."""
    return {
        'primary_image': np.ascontiguousarray(obs_packed['primary_image']),
        'wrist_image':   np.ascontiguousarray(obs_packed['wrist_image']),
        'proprio':       np.asarray(obs_packed['proprio'], dtype=np.float32),
    }


def get_kept_joint_addrs(env, kept_names):
    """Return dict: kept_body_name -> (qpos_start, qvel_start) on env's sim.
    Looked up once per env, reused for every replay."""
    problem = _resolve_libero_problem_obj(env)
    sim = problem.sim
    out = {}
    for k in kept_names:
        obj = problem.objects_dict.get(k)
        if obj is None or not getattr(obj, 'joints', None):
            continue
        ap = sim.model.get_joint_qpos_addr(obj.joints[0])
        qp = ap[0] if isinstance(ap, tuple) else int(ap)
        vap = sim.model.get_joint_qvel_addr(obj.joints[0])
        vp = vap[0] if isinstance(vap, tuple) else int(vap)
        out[k] = (qp, vp)
    return out


def record_pos_rollout(env_pos, init_state, pos_stress, prompt, *, num_steps_wait=10):
    """Run one rollout in env_pos alone. Returns (success, env_steps, records).
    Each record stores:
      pos_qpos / pos_qvel / pos_time : raw arrays from env_pos.sim.data
      obs_stored                      : primary/wrist/proprio dict for the NPZ
    Recording qpos/qvel arrays (instead of the full LIBERO flat sim_state)
    lets pass 2 build env_neg's state without touching env_pos again —
    avoiding env_pos.set_init_state(mid_rollout_state), which appears to
    cause C-side instability in LIBERO/MuJoCo on long jobs."""
    env_pos.reset()
    obs = pos_stress.set_init_state(env_pos, init_state)
    for _ in range(num_steps_wait):
        obs, _, _, _ = env_pos.step(get_libero_dummy_action(cfg.model_family))

    pos_problem = _resolve_libero_problem_obj(env_pos)
    pos_sim = pos_problem.sim

    queue = deque(maxlen=cfg.num_open_loop_steps)
    records = []
    success = False
    t = 0
    while t < max_env_steps:
        if not queue:
            obs_packed = prepare_observation(obs, resize_size=224, flip_images=cfg.flip_images)
            records.append({
                'pos_qpos':   np.asarray(pos_sim.data.qpos, dtype=np.float64).copy(),
                'pos_qvel':   np.asarray(pos_sim.data.qvel, dtype=np.float64).copy(),
                'pos_time':   float(pos_sim.data.time),
                'obs_stored': _store_obs(obs_packed),
            })
            actions = policy_fn(obs_packed, prompt)
            for a in actions[:cfg.num_open_loop_steps]:
                queue.append(np.asarray(a, dtype=np.float32))
        a = queue.popleft()
        obs, _, done, _ = env_pos.step(a.tolist())
        if done:
            success = True
            break
        t += 1
    return success, t + num_steps_wait, records


def expand_recorded_pos_to_neg(pos_qpos, pos_qvel, pos_time, init_state_flat,
                                pos_addrs, neg_addrs, neg_problem, n_kept):
    """Build env_neg flat state from recorded pos qpos/qvel arrays + init_state
    for the removed-body slots. Independent of env_pos liveness."""
    neg_sim = neg_problem.sim
    neg_nq, neg_nv = int(neg_sim.model.nq), int(neg_sim.model.nv)
    robot_nq = len(pos_qpos) - 7 * n_kept
    robot_nv = len(pos_qvel) - 6 * n_kept
    expected_neg_len = 1 + neg_nq + neg_nv
    if len(init_state_flat) != expected_neg_len:
        raise ValueError(
            f'init_state size {len(init_state_flat)} != expected {expected_neg_len}'
        )

    out = np.asarray(init_state_flat, dtype=np.float64).copy()
    out[0] = pos_time
    out[1 : 1 + robot_nq] = pos_qpos[:robot_nq]
    out[1 + neg_nq : 1 + neg_nq + robot_nv] = pos_qvel[:robot_nv]

    for k, (qp, vp) in pos_addrs.items():
        if k not in neg_addrs:
            continue
        qn, vn = neg_addrs[k]
        out[1 + qn : 1 + qn + 7] = pos_qpos[qp : qp + 7]
        out[1 + neg_nq + vn : 1 + neg_nq + vn + 6] = pos_qvel[vp : vp + 6]
    return out


def render_pos_records_in_neg(env_neg, init_state, records,
                              pos_addrs, neg_addrs, n_kept):
    """For each captured record, build the env_neg flat state from recorded
    pos arrays and set env_neg. Returns a list of stored obs dicts, paired
    row-for-row with records."""
    neg_problem = _resolve_libero_problem_obj(env_neg)
    neg_stored = []
    for rec in records:
        flat = expand_recorded_pos_to_neg(
            rec['pos_qpos'], rec['pos_qvel'], rec['pos_time'],
            init_state, pos_addrs, neg_addrs, neg_problem, n_kept,
        )
        obs_neg = env_neg.set_init_state(flat)
        obs_neg_packed = prepare_observation(obs_neg, resize_size=224, flip_images=cfg.flip_images)
        neg_stored.append(_store_obs(obs_neg_packed))
    return neg_stored


def rollout_collect_neg_drives(
    init_state, env_neg, env_pos, pos_stress, neg_stress, prompt,
    *, num_steps_wait=10,
):
    """Neg-drives paired rollout: env_neg steps, env_pos is state-injected per
    inference. Returns (success, env_steps, neg_stored_list, pos_stored_list).
    (Pos-drives uses record_pos_rollout + render_pos_records_in_neg instead.)"""
    env_neg.reset()
    obs_driver = neg_stress.set_init_state(env_neg, init_state)
    pos_stress.set_init_state(env_pos, init_state)

    for _ in range(num_steps_wait):
        obs_driver, _, _, _ = env_neg.step(get_libero_dummy_action(cfg.model_family))

    queue = deque(maxlen=cfg.num_open_loop_steps)
    neg_inputs, pos_inputs = [], []
    success = False
    t = 0
    while t < max_env_steps:
        if not queue:
            neg_packed = prepare_observation(obs_driver, resize_size=224, flip_images=cfg.flip_images)
            driver_state_flat = env_neg.get_sim_state()
            obs_replay = pos_stress.set_init_state(env_pos, driver_state_flat)
            pos_packed = prepare_observation(obs_replay, resize_size=224, flip_images=cfg.flip_images)
            neg_inputs.append(_store_obs(neg_packed))
            pos_inputs.append(_store_obs(pos_packed))
            actions = policy_fn(neg_packed, prompt)
            for a in actions[:cfg.num_open_loop_steps]:
                queue.append(np.asarray(a, dtype=np.float32))
        a = queue.popleft()
        obs_driver, _, done, _ = env_neg.step(a.tolist())
        if done:
            success = True
            break
        t += 1
    return success, t + num_steps_wait, neg_inputs, pos_inputs


## 5. Run all 3 configs — 2-pass per config

Per config:

1. Build env_pos only (matches 10_object_pair_selected's single-env setup —
   env_neg is built *after* pos-drives finishes to avoid perturbing env_pos's
   rendered obs via shared OpenGL / MuJoCo state).
2. **Pass 1 — pos-drives reproduction**: run `record_pos_rollout` for each
   episode in the SUCCESS.mp4 list. The policy is deterministic given
   `(obs, desc)` (every call uses seed=1, fresh numpy RandomState for the
   initial noise, fresh torch.Generator for scheduler steps; no global RNG
   state crosses calls). So with env_pos alone and the same init_state, we
   should reproduce the stress test's trajectory exactly.
3. Build env_neg.
4. **Pass 2a — pos-drives replay**: for each success-list episode that
   reproduced as successful in pass 1, replay its recorded pos qpos/qvel
   into env_neg via `render_pos_records_in_neg`. → rows with `drive_source=1`.
5. **Pass 2b — neg-drives**: run `rollout_collect_neg_drives` for each
   success-list episode. → rows with `drive_source=0`.
6. Close env_neg, env_pos. Move to next config.

Episodes that fail to reproduce as successful in pass 1 are skipped from
pos-drives (their captured trajectory wouldn't match the stress test) and
listed in the manifest's `episodes_missing_success` field for visibility.


In [ ]:
import sys as _sys
import torch as _torch  # for empty_cache between rollouts

def _log(msg):
    """Write progress directly to stderr, bypassing Jupyter's cell-output
    buffering so we can see per-episode progress live in the SLURM .err log."""
    _sys.stderr.write(f'[{time.strftime("%H:%M:%S")}] {msg}\n')
    _sys.stderr.flush()


def _checkpoint_state(suffix):
    """Dump current accumulators + manifest snapshots to a per-config NPZ +
    JSON so a later crash doesn't lose finished configs. Writes to OUT_DIR."""
    if not all_episode_idx:
        return
    ck = OUT_DIR / f'checkpoint_{suffix}.npz'
    np.savez_compressed(
        ck,
        pos_primary=np.stack(all_pos_primary, axis=0),
        pos_wrist=np.stack(all_pos_wrist, axis=0),
        pos_proprio=np.stack(all_pos_proprio, axis=0),
        neg_primary=np.stack(all_neg_primary, axis=0),
        neg_wrist=np.stack(all_neg_wrist, axis=0),
        neg_proprio=np.stack(all_neg_proprio, axis=0),
        episode_idx=np.asarray(all_episode_idx, dtype=np.int32),
        inference_idx=np.asarray(all_inference_idx, dtype=np.int32),
        drive_source=np.asarray(all_drive_source, dtype=np.int32),
        config_idx=np.asarray(all_config_idx, dtype=np.int32),
    )
    ck_json = OUT_DIR / f'checkpoint_{suffix}.json'
    ck_json.write_text(json.dumps({
        'rollout_summaries': rollout_summaries,
        'config_manifests':  config_manifests,
    }, indent=2))
    _log(f'wrote checkpoint {ck} ({ck.stat().st_size/1e6:.1f} MB)')


all_pos_primary, all_pos_wrist, all_pos_proprio = [], [], []
all_neg_primary, all_neg_wrist, all_neg_proprio = [], [], []
all_episode_idx, all_inference_idx, all_drive_source, all_config_idx = [], [], [], []
rollout_summaries = []
config_manifests = []

DRIVE_SOURCES = (
    ('neg_drives', 0),
    ('pos_drives', 1),
)

_log('=== run loop starting ===')

for scene in SCENE_CONFIGS:
    cfg_idx        = scene['cfg_idx']
    name_hint      = scene['name_hint']
    goal_pair      = scene['goal_pair']
    prompt         = scene['prompt']
    target_success = list(scene['episodes'])
    target_set     = set(target_success)

    print(f'\n========== cfg{cfg_idx}  {name_hint}  prompt={prompt!r}')
    print(f'  stress-test successes : {target_success}')
    _log(f'cfg{cfg_idx} {name_hint} START  successes={target_success}')

    pos_stress = SceneRetargetTask(
        replacements=(),
        keep_only=(goal_pair[0], goal_pair[1], CONTAINER),
        goal_pair=goal_pair,
        container_key=CONTAINER,
        prompt=prompt,
        name_hint=f'{name_hint}__pos',
    )
    neg_stress = SceneRetargetTask(
        replacements=(),
        keep_only=None,
        goal_pair=goal_pair,
        container_key=CONTAINER,
        prompt=prompt,
        name_hint=f'{name_hint}__neg',
    )

    # ===== Build env_pos alone (matches 10_object_pair_selected) =====
    _log(f'cfg{cfg_idx} building env_pos')
    pos_task = pos_stress.transform_task(task, output_dir=OUT_DIR)
    env_pos, base_task_desc_pos = get_libero_env(pos_task, 'cosmos', resolution=RESOLUTION)
    print(f'  env_pos kept: {list(pos_stress._final_order)}')
    pos_addrs = get_kept_joint_addrs(env_pos, pos_stress._final_order)
    n_kept    = len(pos_stress._final_order)
    _log(f'cfg{cfg_idx} env_pos ready  kept={list(pos_stress._final_order)}')

    # ===== Pass 1: re-run pos-drives on the success-list episodes =====
    # get_action is deterministic given (obs, desc) — seed=1, randomize_seed=False
    # by default, with a fresh np.RandomState(seed) for initial noise and a fresh
    # torch.Generator for scheduler steps. No global RNG carries between calls,
    # so we can replay just the success episodes (not all 30) and still match
    # the stress test bit-for-bit, as long as env_pos is built without env_neg
    # alongside it.
    print(f'  --- pass 1: env_pos-only rollouts on success-list ({len(target_success)} episodes) ---')
    per_episode_records   = {}
    per_episode_success   = {}
    per_episode_env_steps = {}
    for ep in target_success:
        _log(f'cfg{cfg_idx} pass1 ep={ep} start')
        t0 = time.time()
        success, env_steps, records = record_pos_rollout(
            env_pos, init_states[ep], pos_stress, prompt,
        )
        dt = time.time() - t0
        per_episode_success[ep]   = bool(success)
        per_episode_env_steps[ep] = int(env_steps)
        per_episode_records[ep]   = records
        tag = 'SUCCESS' if success else 'FAILURE'
        print(f'    ep {ep:2d}*  {tag:7s}  steps={env_steps:4d}  inferences={len(records):3d}  {dt:6.1f}s')
        _log(f'cfg{cfg_idx} pass1 ep={ep} {tag} steps={env_steps} inf={len(records)} {dt:.1f}s')
        _torch.cuda.empty_cache()

    reproduced_succ = {ep for ep in target_set if per_episode_success[ep]}
    matched_set     = target_set & reproduced_succ
    extras_set      = set()  # we didn't run non-target episodes, so no extras to report
    missing_set     = target_set - reproduced_succ
    print(f'  pass 1 result: matched {len(matched_set)}/{len(target_set)} stress-test successes')
    if missing_set:
        print(f'    missing (succeeded in stress test, not here): {sorted(missing_set)}')

    # ===== Build env_neg =====
    _log(f'cfg{cfg_idx} building env_neg')
    neg_task = neg_stress.transform_task(task, output_dir=OUT_DIR)
    env_neg, base_task_desc_neg = get_libero_env(neg_task, 'cosmos', resolution=RESOLUTION)
    print(f'  env_neg kept: {list(neg_stress._final_order)}')
    neg_addrs = get_kept_joint_addrs(env_neg, pos_stress._final_order)
    _log(f'cfg{cfg_idx} env_neg ready')

    config_manifests.append({
        'cfg_idx': cfg_idx,
        'name_hint': name_hint,
        'goal_pair': list(goal_pair),
        'prompt': prompt,
        'episodes_target_success': sorted(target_set),
        'episodes_reproduced_success': sorted(matched_set),
        'episodes_extra_success': sorted(extras_set),
        'episodes_missing_success': sorted(missing_set),
        'positive_stress': pos_stress.manifest(),
        'negative_stress': neg_stress.manifest(),
        'env_base_task_desc_pos': base_task_desc_pos,
        'env_base_task_desc_neg': base_task_desc_neg,
    })

    # ===== Pass 2a: pos-drives replay (only matched-success episodes) =====
    emit_eps = sorted(matched_set)
    print(f'  --- pass 2a: pos_drives (drive_source=1) replay over {len(emit_eps)} matched episodes ---')
    _log(f'cfg{cfg_idx} pass2a START replay {len(emit_eps)} episodes')
    for ep in emit_eps:
        _log(f'cfg{cfg_idx} pass2a ep={ep} replay start')
        records = per_episode_records[ep]
        t0 = time.time()
        neg_stored = render_pos_records_in_neg(
            env_neg, init_states[ep], records, pos_addrs, neg_addrs, n_kept,
        )
        dt = time.time() - t0
        assert len(neg_stored) == len(records)

        for inf_idx, (rec, n_st) in enumerate(zip(records, neg_stored)):
            p_st = rec['obs_stored']
            all_neg_primary.append(n_st['primary_image'])
            all_neg_wrist.append(n_st['wrist_image'])
            all_neg_proprio.append(n_st['proprio'])
            all_pos_primary.append(p_st['primary_image'])
            all_pos_wrist.append(p_st['wrist_image'])
            all_pos_proprio.append(p_st['proprio'])
            all_episode_idx.append(ep)
            all_inference_idx.append(inf_idx)
            all_drive_source.append(1)
            all_config_idx.append(cfg_idx)

        n_inf = len(records)
        print(f'    ep {ep:2d}  REPLAYED  inferences={n_inf:3d}  {dt:6.1f}s')
        _log(f'cfg{cfg_idx} pass2a ep={ep} replayed inf={n_inf} {dt:.1f}s')
        rollout_summaries.append({
            'cfg_idx': cfg_idx,
            'name_hint': name_hint,
            'drive_source': 1,
            'drive_name': 'pos_drives',
            'episode': ep,
            'success': True,
            'env_steps': per_episode_env_steps[ep],
            'n_inferences': n_inf,
            'wall_time_s': dt,
            'reproduction_source': 'pass1_record_replay',
        })

    # Free pass-1 records before neg-drives (they're already emitted to the global lists)
    per_episode_records.clear()
    _torch.cuda.empty_cache()

    # ===== Pass 2b: neg-drives (env_neg drives, env_pos state-injected) =====
    print(f'  --- pass 2b: neg_drives (drive_source=0) over {len(target_success)} target episodes ---')
    _log(f'cfg{cfg_idx} pass2b START neg-drives over {len(target_success)} episodes')
    for ep in target_success:
        _log(f'cfg{cfg_idx} pass2b ep={ep} start')
        t0 = time.time()
        success, env_steps, neg_inputs, pos_inputs = rollout_collect_neg_drives(
            init_states[ep], env_neg, env_pos, pos_stress, neg_stress, prompt,
        )
        dt = time.time() - t0
        assert len(neg_inputs) == len(pos_inputs)

        for inf_idx, (n_rec, p_rec) in enumerate(zip(neg_inputs, pos_inputs)):
            all_neg_primary.append(n_rec['primary_image'])
            all_neg_wrist.append(n_rec['wrist_image'])
            all_neg_proprio.append(n_rec['proprio'])
            all_pos_primary.append(p_rec['primary_image'])
            all_pos_wrist.append(p_rec['wrist_image'])
            all_pos_proprio.append(p_rec['proprio'])
            all_episode_idx.append(ep)
            all_inference_idx.append(inf_idx)
            all_drive_source.append(0)
            all_config_idx.append(cfg_idx)

        tag = 'SUCCESS' if success else 'FAILURE'
        n_inf = len(neg_inputs)
        print(f'    ep {ep:2d}  {tag:7s}  steps={env_steps:4d}  inferences={n_inf:3d}  {dt:6.1f}s')
        _log(f'cfg{cfg_idx} pass2b ep={ep} {tag} steps={env_steps} inf={n_inf} {dt:.1f}s')
        rollout_summaries.append({
            'cfg_idx': cfg_idx,
            'name_hint': name_hint,
            'drive_source': 0,
            'drive_name': 'neg_drives',
            'episode': ep,
            'success': bool(success),
            'env_steps': int(env_steps),
            'n_inferences': n_inf,
            'wall_time_s': dt,
        })
        _torch.cuda.empty_cache()

    env_neg.close()
    env_pos.close()
    _torch.cuda.empty_cache()
    print(f'  cfg{cfg_idx} done')
    _log(f'cfg{cfg_idx} DONE total_rows={len(all_episode_idx)}')
    _checkpoint_state(f'cfg{cfg_idx}')

print()
print(f'total paired rows: {len(all_episode_idx)}')


## 6. Save paired NPZs

Row `i` in `positive.npz` and row `i` in `negative.npz` share the same MuJoCo
state at capture time — paired by construction. Both files carry the same
`(episode_idx, inference_idx, drive_source, config_idx)` columns, plus a
`prompts` lookup array (one entry per cfg, indexed by `config_idx`).


In [ ]:
prompts_lookup = np.array([s['prompt'] for s in SCENE_CONFIGS], dtype=object)
name_hints_lookup = np.array([s['name_hint'] for s in SCENE_CONFIGS], dtype=object)


def _stack_save(out_npz, primary_list, wrist_list, proprio_list):
    primary_arr   = np.stack(primary_list, axis=0)
    wrist_arr     = np.stack(wrist_list,   axis=0)
    proprio_arr   = np.stack(proprio_list, axis=0)
    episode_arr   = np.asarray(all_episode_idx,   dtype=np.int32)
    inference_arr = np.asarray(all_inference_idx, dtype=np.int32)
    drive_arr     = np.asarray(all_drive_source,  dtype=np.int32)
    config_arr    = np.asarray(all_config_idx,    dtype=np.int32)
    np.savez_compressed(
        out_npz,
        primary_images=primary_arr,
        wrist_images=wrist_arr,
        proprios=proprio_arr,
        episode_idx=episode_arr,
        inference_idx=inference_arr,
        drive_source=drive_arr,
        config_idx=config_arr,
        prompts=prompts_lookup,
        name_hints=name_hints_lookup,
    )
    size_mb = out_npz.stat().st_size / 1e6
    print(f'wrote {out_npz}  ({size_mb:.1f} MB)')
    print(f'  primary_images: {primary_arr.shape}  dtype={primary_arr.dtype}')
    print(f'  wrist_images:   {wrist_arr.shape}  dtype={wrist_arr.dtype}')
    print(f'  proprios:       {proprio_arr.shape}  dtype={proprio_arr.dtype}')
    return primary_arr.shape[0]


n_pos = _stack_save(POSITIVE_NPZ, all_pos_primary, all_pos_wrist, all_pos_proprio)
n_neg = _stack_save(NEGATIVE_NPZ, all_neg_primary, all_neg_wrist, all_neg_proprio)
assert n_pos == n_neg


# Also emit per-config (pos, neg) NPZ pairs under <name_hint>/ subdirs so
# notebooks/lqr/svd/run_partition_svd_pairs_no_action.sh can run unchanged
# (single-prompt assumption). Each subdir gets a prompt.txt sidecar with
# the matching prompt for that scene.
_PER_CFG_KEYS = ('primary_images', 'wrist_images', 'proprios',
                 'episode_idx',   'inference_idx', 'drive_source',
                 'config_idx')

def _slice_to_dict(arrs_dict, mask):
    return {k: arrs_dict[k][mask] for k in _PER_CFG_KEYS if k in arrs_dict}

_unified_pos = dict(np.load(POSITIVE_NPZ, allow_pickle=True))
_unified_neg = dict(np.load(NEGATIVE_NPZ, allow_pickle=True))
_ci = _unified_pos['config_idx']

print()
print('--- per-config NPZ subdirs (SVD-script compatible) ---')
for s in SCENE_CONFIGS:
    cfg_idx   = s['cfg_idx']
    name_hint = s['name_hint']
    prompt    = s['prompt']
    mask = (_ci == cfg_idx)
    n_rows = int(mask.sum())
    if n_rows == 0:
        print(f'  cfg{cfg_idx} {name_hint}: 0 rows — SKIPPED')
        continue
    sub_dir = OUT_DIR / name_hint
    sub_dir.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(sub_dir / 'positive.npz', **_slice_to_dict(_unified_pos, mask))
    np.savez_compressed(sub_dir / 'negative.npz', **_slice_to_dict(_unified_neg, mask))
    (sub_dir / 'prompt.txt').write_text(prompt + '\n')
    pos_size = (sub_dir / 'positive.npz').stat().st_size / 1e6
    neg_size = (sub_dir / 'negative.npz').stat().st_size / 1e6
    print(f'  cfg{cfg_idx} {name_hint}: {n_rows} rows  pos={pos_size:.1f}MB neg={neg_size:.1f}MB')
del _unified_pos, _unified_neg

pos_proprio = np.stack(all_pos_proprio, axis=0)
neg_proprio = np.stack(all_neg_proprio, axis=0)
max_dproprio = float(np.max(np.abs(pos_proprio - neg_proprio)))
print(f'paired proprio max |diff|: {max_dproprio:.3e}  (should be ~0)')

config_arr_all = np.asarray(all_config_idx,  dtype=np.int32)
drive_arr_all  = np.asarray(all_drive_source, dtype=np.int32)
for s in SCENE_CONFIGS:
    for drv_name, drv_code in DRIVE_SOURCES:
        n = int(((config_arr_all == s['cfg_idx']) & (drive_arr_all == drv_code)).sum())
        print(f'  cfg{s["cfg_idx"]} {drv_name:11s}: {n} rows')


## 7. Visualize a few paired examples per config

A few evenly-spaced inference rows from one episode per `(config, drive_source)`,
side-by-side (neg primary | pos primary | neg wrist | pos wrist). Within a row
the two scenes should differ only in distractor presence.


In [ ]:
import matplotlib.pyplot as plt

VIZ_SAMPLES_PER_GROUP = 2

episode_arr_all   = np.asarray(all_episode_idx,    dtype=np.int32)
inference_arr_all = np.asarray(all_inference_idx,  dtype=np.int32)


def _pick_rows(cfg_idx, drive_code, episode, n_samples):
    mask = (
        (config_arr_all == cfg_idx)
        & (drive_arr_all == drive_code)
        & (episode_arr_all == episode)
    )
    rows = np.where(mask)[0]
    if len(rows) == 0:
        return np.array([], dtype=int)
    if len(rows) <= n_samples:
        return rows
    return rows[np.linspace(0, len(rows) - 1, n_samples).astype(int)]


selected = []
for s in SCENE_CONFIGS:
    cfg_idx = s['cfg_idx']
    ep = s['episodes'][0]
    for drv_name, drv_code in DRIVE_SOURCES:
        for i in _pick_rows(cfg_idx, drv_code, ep, VIZ_SAMPLES_PER_GROUP):
            selected.append((s['name_hint'], cfg_idx, drv_name, drv_code, ep, int(i)))

if not selected:
    print('no rows to visualize')
else:
    n_rows = len(selected)
    fig, axes = plt.subplots(n_rows, 4, figsize=(13, 3.0 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    for row_i, (nh, cfg_idx, drv_name, drv_code, ep, i) in enumerate(selected):
        inf_idx = int(inference_arr_all[i])
        dproprio = float(np.max(np.abs(all_pos_proprio[i] - all_neg_proprio[i])))
        tag = f'cfg{cfg_idx} {drv_name}  ep{ep:2d}  inf{inf_idx:3d}'

        axes[row_i, 0].imshow(all_neg_primary[i])
        axes[row_i, 0].set_title(f'{tag}\nneg primary (cluttered)', fontsize=8)
        axes[row_i, 0].axis('off')
        axes[row_i, 1].imshow(all_pos_primary[i])
        axes[row_i, 1].set_title(f'{nh}\npos primary (uncluttered)', fontsize=8)
        axes[row_i, 1].axis('off')
        axes[row_i, 2].imshow(all_neg_wrist[i])
        axes[row_i, 2].set_title('neg wrist', fontsize=8)
        axes[row_i, 2].axis('off')
        axes[row_i, 3].imshow(all_pos_wrist[i])
        axes[row_i, 3].set_title(f'pos wrist  |dproprio|_inf={dproprio:.1e}', fontsize=8)
        axes[row_i, 3].axis('off')

    plt.tight_layout()
    plt.show()


## 8. Manifest


In [ ]:
manifest = {
    'suite': SUITE_NAME,
    'task_id': TASK_ID,
    'resolution': RESOLUTION,
    'container_key': CONTAINER,
    'rollouts_root': str(ROLLOUTS_ROOT),
    'pairing': (
        'row i in positive.npz and row i in negative.npz share the same MuJoCo '
        'state at capture time. drive_source distinguishes which env stepped: '
        '0 = env_neg drove, env_pos was state-injected via '
        'SceneRetargetTask.set_init_state; 1 = env_pos drove, env_neg was '
        'state-injected via _expand_pos_to_neg (kept-body qpos via joint addrs, '
        'removed-body slots from the episode\'s saved libero init_state).'
    ),
    'image_layout': 'HWC uint8, flip_images=True applied at capture time (same as get_action input)',
    'proprio_layout': 'concat(robot0_gripper_qpos[2], robot0_eef_pos[3], robot0_eef_quat[4]) -> shape (9,) float32',
    'episode_selection': (
        'For each config, only libero init-state indices whose uncluttered '
        'rollout under 10_object_pair_selected produced ep##--SUCCESS.mp4 are '
        'used. Parsed from filenames at notebook run time.'
    ),
    'drive_sources': [
        {'code': 0, 'name': 'neg_drives', 'desc': 'env_neg drives; env_pos is state-injected (sliced)'},
        {'code': 1, 'name': 'pos_drives', 'desc': 'env_pos drives; env_neg is state-injected (expanded via init_state)'},
    ],
    'sets': {
        'positive': {
            'out_npz': str(POSITIVE_NPZ),
            'role': 'uncluttered (keep-only) render at every captured pose',
        },
        'negative': {
            'out_npz': str(NEGATIVE_NPZ),
            'role': 'cluttered (full 8-object) render at every captured pose',
        },
    },
    'configs': config_manifests,
    'rollouts': rollout_summaries,
    'paired_proprio_max_abs_diff': max_dproprio,
    'positive_drives_caveat': (
        'in positive-drives rows (drive_source=1), the removed distractors in '
        'the rendered negative image are at their initial poses (taken from '
        'the libero saved init_state). They never physically interacted with '
        'the robot, because the robot was actually stepping in env_pos where '
        'those objects do not exist. That is an acceptable artifact for the '
        'visual-content semantics this dataset targets.'
    ),
}
MANIFEST_JSON.write_text(json.dumps(manifest, indent=2))
print(f'wrote {MANIFEST_JSON}')


## 9. Summary


In [ ]:
for s in SCENE_CONFIGS:
    cfg_idx = s['cfg_idx']
    for drv_name, drv_code in DRIVE_SOURCES:
        subset = [r for r in rollout_summaries
                  if r['cfg_idx'] == cfg_idx and r['drive_source'] == drv_code]
        n_succ = sum(r['success'] for r in subset)
        n_inf = sum(r['n_inferences'] for r in subset)
        print(f"  cfg{cfg_idx} {drv_name:11s}  success={n_succ}/{len(subset)}  rows={n_inf}")
print(f'  total paired rows           : {len(all_episode_idx)}')
print(f'  paired proprio max |diff|   : {max_dproprio:.3e}')
print(f'  positive (uncluttered) : {POSITIVE_NPZ.resolve()}')
print(f'  negative (cluttered)   : {NEGATIVE_NPZ.resolve()}')
print(f'  manifest               : {MANIFEST_JSON.resolve()}')
